# Análisis exploratorio de datos — Bienestar estudiantil

Este notebook organiza el trabajo en dos bloques:

1. **Comprensión general** — estructura del dataset, dimensiones, tipos de variables y primeras observaciones.
2. **Limpieza básica** — identificación y tratamiento de valores faltantes, duplicados, ruido en texto e inconsistencias en variables numéricas.

In [ ]:
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
%pip install -q openpyxl

In [ ]:
df1 = pd.read_excel(
    r"Data\Consolidado - Información de Caracterización (2013 - 2025) (VE) (26.1).xlsx"
)
df2 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencia (2015-2022) (VE) (26.1).xlsx"
)
df3 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencias (2023 - 20251) (VE) (26.1).xlsx"
)

## 1. Comprensión general

Dimensión del problema, tipos de datos, muestra de filas, estadísticos de variables numéricas, cardinalidad en texto y observaciones iniciales **sobre los datos tal como vienen del archivo** (antes de la limpieza de la sección 2).

In [ ]:
def resumen_comprension(df, nombre):
    print("\n" + "=" * 72)
    print(f" {nombre}")
    print("=" * 72)
    print(f"Dimensiones (filas × columnas): {df.shape[0]:,} × {df.shape[1]}")
    mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memoria aproximada (deep): {mem_mb:.2f} MiB\n")
    print("Tipos de datos por columna:")
    print(df.dtypes.to_string())
    print("\nConteo de valores no nulos por columna:")
    print(df.count().to_string())
    print("\nPrimeras filas:")
    print(df.head(3).to_string())
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        print("\nResumen estadístico (variables numéricas):")
        print(df[num_cols].describe().T.to_string())
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        print("\nCardinalidad — valores únicos en columnas de texto (orden descendente):")
        print(df[obj_cols].nunique(dropna=True).sort_values(ascending=False).to_string())
    print("\n--- Primeras observaciones ---")
    print(
        "- Compare el tamaño de cada tabla con el proceso que representa (caracterización vs. asistencia).\n"
        "- Los tipos `object` suelen ser texto; revise si algún identificador numérico debería tratarse como categoría.\n"
        "- Muchos nulos en columnas de doble programa / secundario suelen ser esperables (no aplica a todos).\n"
        "- Los promedios académicos (PAM/PCP) deberían alinearse con la escala institucional (p. ej. 0–5)."
    )


for etiqueta, d in [
    ("Dataset 1 — Caracterización (2013–2025)", df1),
    ("Dataset 2 — Asistencia (2015–2022)", df2),
    ("Dataset 3 — Asistencia (2023–2025)", df3),
]:
    resumen_comprension(d, etiqueta)

## 2. Limpieza básica

- **Valores faltantes**: conteo y porcentaje por columna; se documentan; no se imputa sin criterio de negocio (los nulos estructurales se conservan).
- **Duplicados**: filas idénticas en todas las columnas; se eliminan conservando la primera ocurrencia.
- **Ruido en texto**: espacios sobrantes, mayúsculas y tildes inconsistentes; se unifica con normalización Unicode (NFKD, sin marcas combinantes) y nombres de columna estables (`snake_case`).
- **Inconsistencias numéricas**: conteo de notas (PAM/PCP) fuera del intervalo [0, 5] cuando aplique el nombre de columna.

In [ ]:
def limpiar_string(x, espacios_a_guion_bajo=False):
    if not isinstance(x, str):
        return x
    nfkd = unicodedata.normalize("NFKD", x)
    sin_acentos = "".join(c for c in nfkd if unicodedata.category(c) != "Mn")
    s = sin_acentos.lower().strip()
    if espacios_a_guion_bajo:
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
    return s


def limpiar_nombres_columnas(df):
    out = df.copy()
    out.columns = [limpiar_string(c, espacios_a_guion_bajo=True) for c in out.columns]
    return out


def limpiar_df(df):
    out = limpiar_nombres_columnas(df)
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].map(limpiar_string)
    return out


def tabla_faltantes(df):
    n = len(df)
    t = df.isnull().sum()
    p = (t / n * 100).round(2)
    tab = pd.DataFrame({"nulos": t, "pct": p})
    return tab[tab["nulos"] > 0].sort_values("nulos", ascending=False)


def conteo_duplicados_fila_completa(df):
    return int(df.duplicated().sum())


def celdas_texto_con_espacios_extremos(df):
    total = 0
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna()
        total += int(s.map(lambda x: isinstance(x, str) and x != x.strip()).sum())
    return total


def conteo_notas_fuera_rango(df, cmin=0.0, cmax=5.0):
    filas = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        cl = col.lower()
        if "pam" in cl or "pcp" in cl:
            mask = df[col].notna() & ((df[col] < cmin) | (df[col] > cmax))
            n = int(mask.sum())
            if n:
                filas[col] = n
    return filas

In [ ]:
print("Diagnóstico previo a normalizar (datos crudos)\n")

for nombre, d in [
    ("df1 — Caracterización", df1),
    ("df2 — Asistencia 2015–2022", df2),
    ("df3 — Asistencia 2023–2025", df3),
]:
    print("-" * 72)
    print(nombre)
    print("Faltantes (columnas con al menos un nulo):")
    tf = tabla_faltantes(d)
    print(tf.to_string() if len(tf) else "  (ninguno)")
    dup = conteo_duplicados_fila_completa(d)
    print(f"Filas duplicadas (todas las columnas): {dup:,}")
    esp = celdas_texto_con_espacios_extremos(d)
    print(f"Celdas de texto con espacios al inicio/fin: {esp:,}")
    out = conteo_notas_fuera_rango(d)
    print(
        "Notas numéricas fuera de [0, 5] (columnas PAM/PCP):",
        out if out else "ninguna detectada",
    )
    print()

In [ ]:
df1 = limpiar_df(df1)
df2 = limpiar_df(df2)
df3 = limpiar_df(df3)

for d, nombre in [(df1, "df1"), (df2, "df2"), (df3, "df3")]:
    antes = len(d)
    n_dup = d.duplicated().sum()
    d.drop_duplicates(inplace=True)
    despues = len(d)
    print(
        f"{nombre}: filas antes {antes:,} | duplicadas eliminadas {int(n_dup):,} | filas después {despues:,}"
    )

print("\nTras limpieza — muestra df1 (primeras filas):")
print(df1.head(3).to_string())

In [ ]:
tf1 = tabla_faltantes(df1)
print("Faltantes después de limpiar (df1):")
print(tf1.to_string() if len(tf1) else "(ninguno)")
print("\nInconsistencias PAM/PCP tras limpiar (df1):")
print(conteo_notas_fuera_rango(df1) or "ninguna detectada")